In [ ]:
import pandas as pd
import numpy as np

'''PRÉ DEFINIÇÕES INICIAIS PARA ORGANIZAÇÃO DOS DADOS'''

# Arquivo de 2025 para base de aprendizado
df_treino = pd.read_csv("INMET_S_PR_A807_CURITIBA_01-01-2025_A_31-12-2025.CSV", # Lê o arquivo CSV
                        sep=';', # define o ponto e vírgula como separador de colunas
                        encoding='latin-1', # resolve problemas de acentuação comuns em arquivos brasileiros
                        skiprows=8, # pula as linhas iniciais do cabeçalho administrativo
                        decimal=',') # converte números como "15,5" para 15.5 (float)

# Arquivo de 2026 para validar o modelo depois
df_teste = pd.read_csv("INMET_S_PR_A807_CURITIBA_01-01-2026_A_28-02-2026.CSV", sep=';', encoding='latin-1', skiprows=8, decimal=',')

display(df_treino.head())

,Data,Hora UTC,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (Kj/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",TEMPERATURA DO PONTO DE ORVALHO (°C),TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C),TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIREÇÃO HORARIA (gr) (° (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",Unnamed: 19
0,2025/01/01,0000 UTC,0.0,911.1,911.1,910.2,NaN,20.3,18.3,20.4,20.2,18.6,18.1,90.0,88.0,89.0,77.0,5.2,2.0,NaN
1,2025/01/01,0100 UTC,0.0,911.7,911.8,911.1,NaN,20.0,18.1,20.4,20.0,18.6,17.9,89.0,88.0,89.0,84.0,5.5,3.0,NaN
2,2025/01/01,0200 UTC,0.0,911.4,911.7,911.3,NaN,19.7,18.1,20.0,19.6,18.3,18.0,91.0,89.0,90.0,92.0,6.0,2.4,NaN
3,2025/01/01,0300 UTC,0.0,911.3,911.4,911.2,NaN,19.7,18.0,19.8,19.7,18.2,17.9,91.0,90.0,90.0,78.0,5.8,2.7,NaN
4,2025/01/01,0400 UTC,0.0,911.0,911.3,911.0,NaN,19.4,18.0,19.7,19.4,18.1,17.9,92.0,90.0,92.0,77.0,6.3,2.3,NaN


Sistema de Classificação

In [10]:
# Substitui o valor de erro (-9999) por "NaN" nos dois DataFrames
df_treino = df_treino.replace(-9999, np.nan)
df_teste = df_teste.replace(-9999, np.nan)

# Criação da coluna alvo 'CHUVA' (0 ou 1) com base na precipitação antes de excluí-la
df_treino['CHUVA'] = (df_treino['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] > 0).astype(int)
df_teste['CHUVA'] = (df_teste['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] > 0).astype(int)

#Separar apenas as colunas importantes para analisar possibilidade de chuva
col_temp = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'
col_umid = 'UMIDADE RELATIVA DO AR, HORARIA (%)'
col_pres = 'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)'
col_rad  = 'RADIACAO GLOBAL (Kj/m²)'

features = [col_temp, col_umid, col_pres, col_rad]

#Variável alvo original
col_chuva = 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'

In [11]:
# Remove linhas com valores nulos (NaN) em uma cópia para não perder os dados originais (se precisar)
df_treino_limpo = df_treino[features + ['CHUVA']].dropna()
df_teste_limpo = df_teste[features + ['CHUVA']].dropna()

# 2. Separando em X (entradas) e y (alvo)
X_train = df_treino_limpo[features]
y_train = df_treino_limpo['CHUVA']

X_test = df_teste_limpo[features]
y_test = df_teste_limpo['CHUVA']

# 3. Aplicando o Escalonamento (StandardScaler)
scaler = StandardScaler()

# O scaler "aprende" a média e desvio do TREINO e aplica em ambos
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Verificação de segurança
print(f"Total de registros para treino: {len(X_train_scaled)}")
print(f"Total de registros para teste: {len(X_test_scaled)}")

Total de registros para treino: 4930
Total de registros para teste: 936
